# 05. Macro F1 0.84 도전 

**이 노트북은 혼자 돌아간다.** 04번 노트북을 먼저 실행할 필요 없다.

---

## 04번에서 배운 것

13번 실험이 전부 **0.816 ~ 0.830** 안에 들어왔다. 폭이 폴드 간 흔들림(±0.02)보다도 좁다.
가중치·SMOTE·스태킹·임계값 — 전부 **"7개 클래스를 한 번에 보는 모델 하나"** 라는 구조를
조금씩 흔든 것에 불과했고, 그 구조의 천장이 0.83 근처라는 뜻이다.

그래서 이번에는 **구조를 바꾼다.** 손실의 대부분이 몰린 구역만 떼어내 전용 모델에 맡긴다.

## 앞선 실험에서 발견한 두 가지 (보고서에 쓸 것)

**1. SMOTE는 클래스 가중치를 무력화한다.**
`class_weight="balanced"`는 클래스 개수를 보고 가중치를 계산하는데, SMOTE가 모든 클래스를
같은 개수로 맞춰버리면 가중치가 전부 1.0이 된다. 즉 **아무 가중치도 안 준 것과 동일**하다.
두 기법은 보완재가 아니라 대체재다. 둘 중 하나만 쓴다.

**2. LightGBM의 `subsample`은 혼자 쓰면 무시된다.**
`subsample_freq`가 0이면(기본값) 서브샘플링이 실제로 작동하지 않는다.
그리고 이 데이터에서는 서브샘플링을 **켜면 오히려 손해**였다(0.8301 → 0.8263).
1,941개뿐이라 80%만 쓰면 희귀 클래스가 더 쪼그라들기 때문이다. 그래서 아래 기준 모델은 꺼둔다.


In [ ]:
# ── 0. 설정 ──────────────────────────────────────────────────────
# 필요 시: uv add lightgbm xgboost imbalanced-learn scikit-learn pandas matplotlib

import warnings, time
warnings.filterwarnings("ignore")

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

from sklearn.model_selection import RepeatedStratifiedKFold, StratifiedKFold
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import f1_score, classification_report, confusion_matrix
from scipy.stats import wilcoxon

from lightgbm import LGBMClassifier
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE

SEED = 42
N_SPLITS, N_REPEATS = 5, 3

def find_data(filename="Faults_processed.csv"):
    for base in [Path.cwd(), *Path.cwd().parents]:
        hit = list(base.glob(f"**/{filename}"))
        if hit:
            return hit[0]
    raise FileNotFoundError(f"{filename} 을 찾지 못했습니다.")

DATA_PATH = find_data()
_p   = Path(DATA_PATH).resolve()
ROOT = next((a for a in _p.parents if (a/"README.md").exists() or (a/".git").exists()), _p.parent)
FIG_DIR = ROOT / "figures"; FIG_DIR.mkdir(parents=True, exist_ok=True)

for _n in ["Malgun Gothic", "AppleGothic", "NanumGothic", "DejaVu Sans"]:
    if any(f.name == _n for f in fm.fontManager.ttflist):
        plt.rcParams["font.family"] = _n; break
plt.rcParams["axes.unicode_minus"] = False
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 60)

print("데이터 :", DATA_PATH)
print("그림   :", FIG_DIR)

In [ ]:
# ── 1. 데이터 · 공통 함수 ────────────────────────────────────────
df    = pd.read_csv(DATA_PATH)
y_raw = df.pop("target")
le    = LabelEncoder().fit(y_raw)
y     = le.transform(y_raw)
CLASSES = list(le.classes_)
X     = df.copy()

FOLDS = list(RepeatedStratifiedKFold(n_splits=N_SPLITS, n_repeats=N_REPEATS,
                                     random_state=SEED).split(X, y))

# ── 난수 설정 헬퍼 (★04번 버그 수정) ──
# 파이프라인은 'random_state'가 아니라 's__random_state', 'm__random_state' 를 쓴다.
# 04번 코드는 이걸 놓치고 에러를 조용히 삼켜서, 똑같은 모델 7개를 평균내고 있었다.
def set_all_seeds(m, s):
    keys = [k for k in m.get_params() if k.endswith("random_state")]
    if keys:
        m.set_params(**{k: s for k in keys})
    return m

class SeedBag:
    '''난수만 바꿔 여러 번 학습하고 확률을 평균 — 운에 의한 흔들림을 상쇄'''
    def __init__(self, fn, n=7): self.fn, self.n = fn, n
    def fit(self, Xt, yt):
        self.ms_ = [set_all_seeds(self.fn(), SEED + 1000*s).fit(Xt, yt) for s in range(self.n)]
        return self
    def predict_proba(self, Xe):
        return np.mean([m.predict_proba(Xe) for m in self.ms_], axis=0)
    def predict(self, Xe): return self.predict_proba(Xe).argmax(1)

# ── 기준 모델 (★서브샘플링 끔 — 04번 ⑦과 동일 조건) ──
def lgb_fix(**kw):
    p = dict(n_estimators=300, learning_rate=0.05, num_leaves=31, colsample_bytree=0.8,
             random_state=SEED, n_jobs=-1, verbose=-1)
    p.update(kw)
    return LGBMClassifier(**p)

def smote(): return SMOTE(k_neighbors=4, random_state=SEED)   # Dirtiness 55개 → 이웃 수 작게

def evaluate(make_model, folds=None, weights=None, tag=""):
    folds = FOLDS if folds is None else folds
    sc, t0 = [], time.time()
    for tr, te in folds:
        m = make_model(); m.fit(X.iloc[tr], y[tr])
        P = m.predict_proba(X.iloc[te])
        pred = (P * weights).argmax(1) if weights is not None else P.argmax(1)
        sc.append(f1_score(y[te], pred, average="macro"))
    sc = np.array(sc)
    print(f"{tag:36s} macro F1 = {sc.mean():.4f} ± {sc.std():.4f}   ({time.time()-t0:.0f}초)")
    return sc

def compare(sc, baseline, tag=""):
    d = sc - baseline
    p = wilcoxon(sc, baseline)[1] if np.any(d != 0) else 1.0
    print(f"{tag:36s} 평균차 {d.mean():+.4f}   {len(sc)}폴드 중 {int((d>0).sum())}회 우세   p={p:.4f}")
    return dict(실험=tag, macro_f1=sc.mean(), 표준편차=sc.std(),
                기준대비=d.mean(), 우세폴드=int((d>0).sum()), p값=p)

def tune_weights(proba, y_true, n_round=10, grid=np.linspace(0.4, 4.0, 37)):
    '''좌표상승법으로 클래스별 확률 배율 탐색'''
    w = np.ones(proba.shape[1])
    best = f1_score(y_true, (proba*w).argmax(1), average="macro")
    for _ in range(n_round):
        moved = False
        for k in range(len(w)):
            cur = w[k]
            for g in grid:
                w[k] = g
                s = f1_score(y_true, (proba*w).argmax(1), average="macro")
                if s > best + 1e-6: best, cur, moved = s, g, True
            w[k] = cur
        if not moved: break
    return w, best

RESULTS = []
print(f"{X.shape[0]}행 × {X.shape[1]}변수 / 클래스 {len(CLASSES)}개 / 폴드 {len(FOLDS)}개")

In [ ]:
# ── 2. 기준선 (여기부터 모든 비교의 출발점) ──────────────────────
base_scores = evaluate(lgb_fix, tag="① 기준선 (LightGBM)")
RESULTS.append(dict(실험="① 기준선", macro_f1=base_scores.mean(), 표준편차=base_scores.std(),
                    기준대비=0.0, 우세폴드=0, p값=np.nan))

## 실험 A. 기반 모델 확정 + 시드 평균 (수정판)

04번에서 가장 좋았던 SMOTE와 클래스 가중치를, 이번엔 **서브샘플링을 끈 동일 조건**에서 다시 잰다.
그리고 04번에서 조용히 실패했던 **시드 평균을 제대로 적용**해 진짜 효과를 확인한다.

시드 평균은 가장 안전한 개선이다. 새 가정도 새 변수도 없이 같은 모델을 난수만 바꿔 여러 번
학습한 뒤 확률을 평균낼 뿐이라, 운에 의한 흔들림만 상쇄된다. 대신 시간이 횟수만큼 늘어난다.

In [ ]:
# ── 3. 실험 A ────────────────────────────────────────────────────
A = {
 "② SMOTE":            lambda: ImbPipeline([("s", smote()), ("m", lgb_fix())]),
 "③ 클래스 가중치":       lambda: lgb_fix(class_weight="balanced"),
 "④ 시드평균 + SMOTE":   lambda: SeedBag(lambda: ImbPipeline([("s", smote()), ("m", lgb_fix())]), 7),
 "⑤ 시드평균 + 가중치":    lambda: SeedBag(lambda: lgb_fix(class_weight="balanced"), 7),
}
for n, f in A.items():
    RESULTS.append(compare(evaluate(f, tag=n), base_scores, n))

# ④ 와 ⑤ 중 좋았던 쪽을 1단계 모델로 채택 (아래 셀들에서 사용)
STAGE1 = lambda: SeedBag(lambda: lgb_fix(class_weight="balanced"), 5)   # ← 결과 보고 수정
print("\n1단계 모델을 위 결과에 맞춰 STAGE1 에 지정했는지 확인할 것")

## 실험 B. 2단계 전문 모델 ★ 이번 노트북의 핵심

**진단.** 7개 클래스 중 네 개(K_Scratch·Z_Scratch·Stains·Dirtiness)는 이미 F1 0.87~0.97로
해결되어 있다. 남은 손실은 거의 전부 **Pastry·Bumps·Other_Faults 세 클래스가 서로 엉킨 구역**
하나에서 나온다. 이 셋이 전체의 약 63%를 차지한다.

**아이디어.** 의사도 1차 진료에서 애매하면 전문의에게 보낸다. 같은 구조다.

- **1단계(일반의)** — 지금처럼 7개 클래스를 전부 본다.
- **2단계(전문의)** — 1단계가 엉킨 세 클래스 중 하나라고 본 샘플만, **그 셋만 학습한 전용 모델**에 다시 묻는다.

전문 모델이 나은 이유는 둘이다. K_Scratch 같은 쉬운 클래스에 신경 쓸 필요가 없어
**세 클래스를 가르는 경계에만 집중**할 수 있고, 셋만 남으면 개수가 비교적 고르게 분포해
불균형 문제도 완화된다.

**주의.** 2단계 모델도 반드시 **학습 폴드 데이터로만** 학습해야 한다. 아래 코드는 바깥 폴드의
학습셋 안에서 모든 것을 끝낸다. 발표에서 "2단계 모델은 어느 데이터로 학습했나요"가 반드시 나온다.

In [ ]:
# ── 4. 엉킨 구역 정의 ────────────────────────────────────────────
HARD_NAMES = ["Pastry", "Bumps", "Other_Faults"]
HARD = [CLASSES.index(c) for c in HARD_NAMES]
EASY = [i for i in range(len(CLASSES)) if i not in HARD]
print("엉킨 구역:", HARD_NAMES, f"→ 전체의 {np.isin(y, HARD).mean():.1%}")
print("안정 구역:", [CLASSES[i] for i in EASY])

In [ ]:
# ── 5. 실험 B: 2단계 전문 모델 ───────────────────────────────────
def specialist(**kw):
    '''엉킨 3종 전용 — 더 깊고 촘촘하게'''
    p = dict(n_estimators=700, learning_rate=0.03, num_leaves=63, min_child_samples=8,
             colsample_bytree=0.7, reg_lambda=1.0, class_weight="balanced",
             random_state=SEED, n_jobs=-1, verbose=-1)
    p.update(kw)
    return LGBMClassifier(**p)

def evaluate_two_stage(stage1_fn, stage2_fn, blend=0.5, use_weights=False,
                       folds=None, tag=""):
    '''blend : 2단계 의견을 얼마나 믿을지 (0=1단계만, 1=2단계만)'''
    folds = FOLDS if folds is None else folds
    sc, t0 = [], time.time()
    remap = {c: i for i, c in enumerate(HARD)}
    for oi, (tr, te) in enumerate(folds):
        Xtr, ytr = X.iloc[tr], y[tr]

        m1 = stage1_fn(); m1.fit(Xtr, ytr)
        P = m1.predict_proba(X.iloc[te]).copy()

        hm = np.isin(ytr, HARD)                       # 학습셋 중 엉킨 3종만
        m2 = stage2_fn(); m2.fit(Xtr[hm], np.array([remap[v] for v in ytr[hm]]))
        P2 = m2.predict_proba(X.iloc[te])

        send = P[:, HARD].sum(1) > P[:, EASY].sum(1)  # 1단계가 '엉킨 구역'이라 본 샘플만
        if send.any():
            mass = P[send][:, HARD].sum(1, keepdims=True)   # 구역 전체 확률량은 보존
            P[np.ix_(send, HARD)] = (1-blend) * P[send][:, HARD] + blend * P2[send] * mass

        if use_weights:          # 배율은 학습셋 내부 OOF에서만 탐색 (누수 방지)
            inner = StratifiedKFold(4, shuffle=True, random_state=SEED + oi)
            oof = np.zeros((len(tr), len(CLASSES)))
            for itr, ite in inner.split(Xtr, ytr):
                mi = stage1_fn(); mi.fit(Xtr.iloc[itr], ytr[itr])
                oof[ite] = mi.predict_proba(Xtr.iloc[ite])
            w, _ = tune_weights(oof, ytr)
        else:
            w = np.ones(len(CLASSES))

        sc.append(f1_score(y[te], (P*w).argmax(1), average="macro"))
    sc = np.array(sc)
    print(f"{tag:36s} macro F1 = {sc.mean():.4f} ± {sc.std():.4f}   ({time.time()-t0:.0f}초)")
    return sc

# 먼저 반영비율을 훑는다 (시드평균 없이 빠르게)
two_stage = {}
for b in [0.3, 0.5, 0.7, 1.0]:
    s = evaluate_two_stage(lambda: lgb_fix(class_weight="balanced"), specialist,
                           blend=b, tag=f"⑥ 2단계 (반영비율 {b})")
    two_stage[b] = s
    RESULTS.append(compare(s, base_scores, f"⑥ 2단계 (반영비율 {b})"))

BEST_BLEND = max(two_stage, key=lambda k: two_stage[k].mean())
print(f"\n가장 좋았던 반영비율: {BEST_BLEND}")

### 반영비율 결과 읽는 법

| 패턴 | 의미 | 다음 행동 |
|---|---|---|
| 비율이 클수록 좋아짐 (1.0 최고) | 전문 모델이 1단계보다 확실히 낫다 | 비율 1.0 채택, 전문 모델을 더 튜닝 |
| 0.5 부근이 최고 | 둘이 서로 보완한다 | 그대로 사용 |
| 0.3에서만 좋음 | 전문 모델이 과적합 중 | `n_estimators`를 300으로 줄이고 `min_child_samples`를 20으로 올려 재시도 |
| 전부 기준선 이하 | 1단계가 구역 판정을 잘못하고 있다 | 아래 '구역 판정 정확도' 셀로 원인 확인 |

In [ ]:
# ── 6. 진단: 1단계의 '구역 판정'은 정확한가 ──────────────────────
# 2단계가 효과 없다면, 애초에 엉뚱한 샘플을 전문의에게 보내고 있을 수 있다.
tr, te = FOLDS[0]
m1 = lgb_fix(class_weight="balanced").fit(X.iloc[tr], y[tr])
P  = m1.predict_proba(X.iloc[te])
send   = P[:, HARD].sum(1) > P[:, EASY].sum(1)
actual = np.isin(y[te], HARD)
print(f"전문의에게 보낸 비율      : {send.mean():.1%}")
print(f"그중 실제로 엉킨 3종인 비율 : {actual[send].mean():.1%}  (정밀도)")
print(f"실제 엉킨 3종 중 보낸 비율 : {send[actual].mean():.1%}  (재현율)")
print("\n→ 정밀도·재현율이 모두 90% 이상이면 구역 판정은 문제 없음")

In [ ]:
# ── 7. 2단계 + 시드평균 결합 ─────────────────────────────────────
final = evaluate_two_stage(
    stage1_fn = lambda: SeedBag(lambda: lgb_fix(class_weight="balanced"), 5),
    stage2_fn = lambda: SeedBag(lambda: specialist(), 5),
    blend     = BEST_BLEND,
    tag       = f"⑦ 2단계({BEST_BLEND}) + 시드평균")
RESULTS.append(compare(final, base_scores, f"⑦ 2단계({BEST_BLEND}) + 시드평균"))

## 실험 C. 임계값 조정 — 깨끗한 조건에서 한 번만

04번의 ⑤·⑩에서 임계값 조정이 아무 효과가 없었다. **원인은 이미 밝혀졌다.**
두 실험 모두 기반 모델에 `class_weight="balanced"`가 들어 있어서, **학습 단계에서 이미
판정선을 희귀 클래스 쪽으로 밀어놓은 상태**였다. 거기에 사후 조정을 더해봐야 겹칠 뿐이다.

그래서 이번엔 **가중치가 전혀 없는 맨 모델** 위에서 한 번만 확인한다.
여기서도 효과가 없으면 "가중치와 임계값 조정은 대체 관계"라는 결론이 확정된다.

In [ ]:
# ── 8. 실험 C: 가중치 없는 모델 + 임계값 조정 ────────────────────
def evaluate_with_weight_tuning(make_model, n_inner=4, folds=None, tag=""):
    folds = FOLDS if folds is None else folds
    sc, wlog, t0 = [], [], time.time()
    for oi, (tr, te) in enumerate(folds):
        Xtr, ytr = X.iloc[tr], y[tr]
        inner = StratifiedKFold(n_inner, shuffle=True, random_state=SEED + oi)
        oof = np.zeros((len(tr), len(CLASSES)))
        for itr, ite in inner.split(Xtr, ytr):
            mi = make_model(); mi.fit(Xtr.iloc[itr], ytr[itr])
            oof[ite] = mi.predict_proba(Xtr.iloc[ite])
        w, _ = tune_weights(oof, ytr); wlog.append(w)
        m = make_model(); m.fit(Xtr, ytr)
        sc.append(f1_score(y[te], (m.predict_proba(X.iloc[te])*w).argmax(1), average="macro"))
    sc = np.array(sc)
    print(f"{tag:36s} macro F1 = {sc.mean():.4f} ± {sc.std():.4f}   ({time.time()-t0:.0f}초)")
    return sc, np.array(wlog)

s, W = evaluate_with_weight_tuning(lgb_fix, tag="⑧ 가중치 없음 + 임계값 조정")
RESULTS.append(compare(s, base_scores, "⑧ 가중치 없음 + 임계값 조정"))
print()
print(pd.DataFrame({"클래스": CLASSES, "평균 배율": W.mean(0).round(2),
                    "샘플 수": np.bincount(y)}).to_string(index=False))
print("\n→ 희귀 클래스에 1보다 큰 배율이 붙었다면 의도대로 작동한 것")

## 결과 정리

In [ ]:
# ── 9. 전체 표 + 그래프 ──────────────────────────────────────────
tbl = pd.DataFrame(RESULTS).round(4).sort_values("macro_f1", ascending=False)
display(tbl)
tbl.to_csv(ROOT / "results_stage2.csv", index=False, encoding="utf-8-sig")

fig, ax = plt.subplots(figsize=(9, 5))
t = tbl.sort_values("macro_f1")
ax.barh(t["실험"], t["macro_f1"], xerr=t["표준편차"], color="#4C78A8", alpha=.85, capsize=3)
ax.axvline(base_scores.mean(), color="crimson", ls="--", lw=1.2, label="기준선")
ax.axvline(0.84, color="green", ls=":", lw=1.5, label="목표 0.84")
ax.set_xlim(0.78, 0.88); ax.set_xlabel("Macro F1 (5-fold × 3회 반복 평균)")
ax.legend(); ax.grid(axis="x", alpha=.3)
plt.tight_layout(); plt.savefig(FIG_DIR / "model_comparison.png", dpi=150); plt.show()

In [ ]:
# ── 10. 개선이 어느 클래스에서 나왔는지 (보고서 8.2절) ───────────
tr, te = FOLDS[0]
Xtr, ytr = X.iloc[tr], y[tr]
remap = {c: i for i, c in enumerate(HARD)}

m1 = SeedBag(lambda: lgb_fix(class_weight="balanced"), 5).fit(Xtr, ytr)
P  = m1.predict_proba(X.iloc[te]).copy()
hm = np.isin(ytr, HARD)
m2 = SeedBag(lambda: specialist(), 5).fit(Xtr[hm], np.array([remap[v] for v in ytr[hm]]))
send = P[:, HARD].sum(1) > P[:, EASY].sum(1)
mass = P[send][:, HARD].sum(1, keepdims=True)
P[np.ix_(send, HARD)] = (1-BEST_BLEND)*P[send][:, HARD] + BEST_BLEND*m2.predict_proba(X.iloc[te])[send]*mass
pred_final = P.argmax(1)
pred_base  = lgb_fix().fit(Xtr, ytr).predict(X.iloc[te])

cmp = pd.DataFrame({"클래스": CLASSES,
                    "기준선 F1": f1_score(y[te], pred_base,  average=None).round(3),
                    "최종 F1":   f1_score(y[te], pred_final, average=None).round(3),
                    "샘플 수":   np.bincount(y[te])})
cmp["변화"] = (cmp["최종 F1"] - cmp["기준선 F1"]).round(3)
display(cmp)

cm = confusion_matrix(y[te], pred_final, normalize="true")
fig, ax = plt.subplots(figsize=(6.5, 5.5))
im = ax.imshow(cm, cmap="Blues", vmin=0, vmax=1)
ax.set_xticks(range(len(CLASSES)), CLASSES, rotation=45, ha="right")
ax.set_yticks(range(len(CLASSES)), CLASSES)
ax.set_xlabel("예측"); ax.set_ylabel("실제")
for i in range(len(CLASSES)):
    for j in range(len(CLASSES)):
        if cm[i, j] > .005:
            ax.text(j, i, f"{cm[i,j]:.2f}", ha="center", va="center",
                    color="white" if cm[i, j] > .5 else "black", fontsize=9)
fig.colorbar(im, shrink=.8)
plt.tight_layout(); plt.savefig(FIG_DIR / "confusion_matrix.png", dpi=150); plt.show()

## 마지막 — 재현성 점검

**최종 구성이 확정된 뒤 딱 한 번만 실행한다.** 결과가 마음에 안 든다고 난수를 바꿔 재시도하면
점검이 아니라 쇼핑이 된다. 미리 정해두고 지킬 것.

지금까지 같은 15개 폴드를 수십 번 들여다보며 방법을 골랐으므로, 그 과정 자체가 폴드에
조금씩 맞춰지는 행위다. 한 번도 쓰지 않은 분할에서 개선폭이 재현되는지 확인해야 한다.

In [ ]:
# ── 11. 재현성 점검 (최종 1회) ───────────────────────────────────
HOLD = list(RepeatedStratifiedKFold(n_splits=5, n_repeats=2,
                                    random_state=777000).split(X, y))

b2 = evaluate(lgb_fix, folds=HOLD, tag="기준선 (새 분할)")
f2 = evaluate_two_stage(
        stage1_fn = lambda: SeedBag(lambda: lgb_fix(class_weight="balanced"), 5),
        stage2_fn = lambda: SeedBag(lambda: specialist(), 5),
        blend     = BEST_BLEND, folds=HOLD, tag="최종 모델 (새 분할)")

print(f"\n새 분할 개선폭   : {f2.mean() - b2.mean():+.4f}")
print(f"원래 분할 개선폭 : {final.mean() - base_scores.mean():+.4f}")
print("두 값이 비슷하면 개선은 진짜, 절반 이하로 줄면 폴드에 과적합된 것")

---

## 반드시 지킬 선

목표 숫자를 정해두고 실험을 반복하면, **점수를 올리는 것**과 **점수가 올라 보이게 만드는 것**이
섞이기 쉽다. 발표에서 치명상을 입는 쪽은 후자다.

**해도 되는 것** — 학습 폴드 안에서 임계값·전문 모델·리샘플링을 모두 끝내고 평가 폴드는 결과만 받는다 ·
난수를 바꿔 여러 번 학습해 평균낸다 · 효과 없던 실험도 전부 표에 남긴다

**하면 안 되는 것** — 평가 데이터를 보고 임계값을 맞춘다 · 여러 난수 중 가장 잘 나온 하나만 보고한다 ·
분할 방식을 바꿔가며 점수가 높은 조합을 고른다 · 어려운 클래스를 말없이 제외한다

마지막 세 개는 전부 "0.84가 안 나와서" 하게 되는 일이다. 미리 정해두면 흔들리지 않는다.

## 다중 비교 문제 — 발표에서 먼저 꺼낼 것

p값 0.05는 "우연히 이럴 확률이 20분의 1"이라는 뜻이다. **실험을 20번 하면 그중 하나쯤
우연히 통과하는 건 당연하다.** 지금 실험 횟수가 그 수준에 가까워지고 있다.

그래서 유의하게 나온 방법은 p값에 기대지 말고 **별도 분할에서 재현되는지로** 판단해야 한다.
발표에서 이걸 먼저 말하면 가장 강한 장면이 된다.

> "다중 비교 문제를 인지하여, 유의하게 나온 방법은 p값에 의존하지 않고 별도 분할에서
> 재현성을 확인했습니다."

## 그래도 0.84가 안 나오면

이 데이터에서 7클래스 macro F1은 보통 0.78~0.84에 모인다. 0.835에서 멈출 가능성이 실제로 있다.
그때 쓸 정직한 카드는 셋이다.

1. **한계를 수치로 보고한다.** Pastry vs Other_Faults를 전용 이진 모델로 구분해도 AUC가
   0.92 부근이다. "현재 변수로는 이 경계를 더 가를 정보가 없다"를 **측정 결과로** 보일 수 있다.
   목표 미달을 변명이 아니라 분석 결과로 만드는 방법이다.
2. **무엇이 있으면 되는지 제시한다.** 결함 이미지 원본, 압연 속도·온도 같은 공정 로그,
   Other_Faults의 세부 라벨 중 하나만 확보되면 이 경계가 열릴 가능성이 높다.
3. **지표를 다시 협의한다.** Other_Faults는 "위 여섯 개 중 어느 것도 아님"에 가까운
   잡동사니 클래스다. 이걸 나머지와 동등하게 평가하는 게 타당한지 자체가 논점이다.
   7클래스 수치를 숨기지 않고 6클래스 macro F1을 **함께** 보고하면 설득력이 올라간다.
